# Image steering: overfit one real asynchronous trajectory

This notebook records an actual image replacement in AsyncLLM, checks differentiable replay, fits one student, then reloads it for **free generation** with the same external event. Teacher forcing, exact token reproduction, and behavioral reproduction are reported separately.

Based on `training-internal-refactor` (`33136ed`) and the text overfit examples in this checkout. Run on **caradhras**, in the project environment. The source is developed locally and synchronized; see `README.md`. Artifacts are reused only from this explicitly selected run directory. Choose a fresh directory to repeat the experiment.

Replay settings are explicit in `SETUP`: Torch attention with query chunks of 64 for parity, training, and checkpoint reload. To validate current automatic selection, set `replay_attention_backend="auto"` and `replay_query_chunk_size=1024` before collecting into a **fresh RUN directory**. Cached reports with missing or different replay settings are rejected. The historical run report describes the earlier execution; the updated notebook has not been executed yet.

## A. Setup

Use one free GPU. Separate child processes release engine/model memory between stages. Sampling is greedy for teacher and student, with a fixed seed; the image event occurs after exactly `update_after` emitted model tokens, excluding input text.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "python/minisgl/training/recording.py").exists()
)
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "python")]
os.environ["PATH"] = str(Path(sys.executable).parent) + ":/usr/local/cuda/bin:" + os.environ["PATH"]
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["MPLBACKEND"] = "Agg"

import json
import subprocess
from io import BytesIO

import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from minisgl.training import Trace
from PIL import Image

from minisgl_examples.training.image_steering import experiment as exp
from minisgl_examples.training.process import isolated


def display_figure(fig):
    # Explicit PNG output also works with Agg in headless notebook kernels.
    with BytesIO() as buffer:
        fig.savefig(buffer, format="png", dpi=160, bbox_inches="tight")
        display(DisplayImage(data=buffer.getvalue()))


TEACHER = "Qwen/Qwen3.5-4B"
STUDENT = "Qwen/Qwen3.5-0.8B"
SETUP = exp.asdict(exp.Setup(replay_attention_backend="torch", replay_query_chunk_size=64))
RUN = ROOT / "training_runs/image_steering"
RUN.mkdir(parents=True, exist_ok=True)
print(
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.used,utilization.gpu", "--format=csv"],
        text=True,
    )
)
print("GPU:", os.environ["CUDA_VISIBLE_DEVICES"], "Run:", RUN)
print(SETUP)


## B. Task: a corrected table cell

MathVista `testmini`, problem 45, from the repository's existing image-correction dataset. Image A says September **20**, image B says **14**; August stays **17**. The rate changes from **+3** to **−3 minutes/month**. Labels are for evaluation only; they never enter inference.

The update contains only image B and a fixed notification that the picture was corrected. It contains no values, answer, or teacher-generated reasoning.

In [ ]:
display(Markdown(exp.QUESTION))
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, side, title in zip(axes, ["before", "after"], ["A: initial", "B: corrected"]):
    ax.imshow(Image.open(exp.ASSETS / f"{side}.png"))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
display_figure(fig)
plt.close(fig)
print("Expected:", exp.task()["answer_before"], "→", exp.task()["answer_after"])
print("External notification:", exp.NOTICE)


## C. Teacher inference and persistence

Prompt, image, and reasoning occupy separate CacheBlocks. The current reasoning view is `[prompt, image, reasoning]`. At the token barrier the pending emitted token is committed under A; that flush is unsupervised. The old image block is freed, B is encoded into a new block conditioned on the prompt, and the view is replaced. Historical reasoning KV is retained. The notification is prefilled; its final logits produce the first post-update token.

This is the image replacement mechanism in `image_async_thoughts_eval.py`, reduced to one generating stream. It does not use the thinker/writer switching probe. All model tokens are selected from real AsyncLLM logits.

In [ ]:
def collect_once(model, name, *, update=True, replacement_image="after"):
    path = RUN / f"{name}.json"
    if not path.exists():
        isolated(
            exp.collect,
            model,
            path,
            SETUP,
            update=update,
            replacement_image=replacement_image,
            log_path=RUN / f"{name}.log",
        )
    result = exp.require_replay_options(exp.load(path), SETUP)
    assert result["model"] == str(model)
    assert result["setup"] == SETUP and result["update"] == update
    assert result["task"] == exp.task()
    assert result.get("replacement_image", "after") == replacement_image
    return path, result


teacher_path, teacher = collect_once(TEACHER, "teacher")
print(teacher["assessment"])
print("Before update:\n", teacher["before_update_text"])
print("After update:\n", teacher["after_update_text"])
assert teacher["assessment"][
    "behavior_reproduced"
], "Teacher failed: choose another example/model before training."


## D. Rollout inspection

Portable format-v2 traces contain requests in execution order: token IDs, write positions, ordered view lengths, image pixels/grids/types, three-axis MRoPE positions, rotary spans, and clear/copy references. They do not contain cached image embeddings or detached numeric KV states. Targets label the row that produced each sampled token; every other row uses `-100`. A global next-token shift would be incorrect here.

In [ ]:
inspection = exp.inspect_rollout(teacher_path)
print(json.dumps(teacher["events"], indent=2))
print(json.dumps(inspection, indent=2))


In [ ]:
trace = Trace.from_dict(teacher["trace"])
tok = __import__("transformers").AutoTokenizer.from_pretrained(TEACHER)
k = SETUP["update_after"]
for i in range(max(0, k - 5), min(len(teacher["token_ids"]), k + 8)):
    row = teacher["sample_rows"][i]
    t = trace.tokens[row]
    print(
        {
            "generated_index": i + 1,
            "phase": teacher["sample_phases"][i],
            "text": tok.decode([teacher["token_ids"][i]]),
            "target_row": row,
            "input_position": t.position,
            "step": t.step,
            "view": t.view,
            "inference_logprob": teacher["logprobs"][row],
        }
    )


## E. Replay parity before updating weights

`replay_trace` applies the actual cache visibility and block-relative MRoPE; attention performs one softmax over each complete view. GDN rebuilds the recurrent computation in view order. Both saved images are re-encoded through the vision tower. Ordinary flattened causal teacher forcing is never used.

First compare teacher inference with teacher replay on the **same weights**, separately before/after replacement. FP16 inference versus replay is numerical, not bitwise; maximum absolute target-logprob error must be ≤0.1. For the student, the same check uses its own baseline trajectory, not the teacher's logprobs.

In [ ]:
teacher_parity_path = RUN / "teacher_parity.json"
if not teacher_parity_path.exists():
    isolated(
        exp.check_replay,
        teacher_path,
        teacher_parity_path,
        dtype="float16",
        log_path=RUN / "teacher_parity.log",
    )
teacher_parity = exp.require_replay_options(exp.load(teacher_parity_path), SETUP)
print({k: v for k, v in teacher_parity["parity"].items() if k != "errors"})
assert teacher_parity["parity"]["passed"]


### Student before training

This is a standalone free-running rollout through AsyncLLM with identical inputs, agent configuration, decoding, token budget, and image-update trigger. No teacher tokens are supplied.

In [ ]:
before_path, before = collect_once(STUDENT, "student_before")
print(before["assessment"])
print(before["text"])


## F. One-trajectory overfit

Reuse the shared chunked loss and `Trace.to("cuda")`; optimize the student's language parameters with AdamW (3e-6) in FP32. The vision tower is frozen for memory, but its outputs for A/B are recomputed in replay. No loss is applied to user/prompt/image/notification inputs or flushes.

The loop checks finite nonzero gradients and a real parameter delta. It evaluates every five updates and stops at 100% teacher-forced token accuracy with ≥0.95 probability for the complete teacher sequence, or at the step budget. This threshold is **not** the free-generation success criterion.

In [ ]:
training_path = RUN / "trained_low_lr/training.json"
if not training_path.exists():
    isolated(
        exp.train,
        teacher_path,
        STUDENT,
        before_path,
        RUN / "trained_low_lr",
        steps=80,
        lr=3e-6,
        log_path=RUN / "training_low_lr.log",
    )
training = exp.require_replay_options(exp.load(training_path), SETUP)
print("Loss:", training["initial"]["loss"], "→", training["final"]["loss"])
print("Accuracy:", training["initial"]["accuracy"], "→", training["final"]["accuracy"])
print("Parameter max delta:", training["tracked_parameter_max_delta"])
assert training["tracked_parameter_max_delta"] > 0
student_parity = exp.require_replay_options(
    exp.load(RUN / "trained_low_lr/student_initial_parity.json"), SETUP
)
print(
    "Student own-trajectory parity:",
    {k: v for k, v in student_parity["parity"].items() if k != "errors"},
)


## G. Evaluation: training reproduction, free generation, before/after

Loss/token probabilities measure teacher forcing. The following reload and free-running runs measure the saved checkpoint. The update remains an external token-count event and is never replaced with teacher reasoning.

In [ ]:
history = training["history"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot([r["step"] for r in history], [r["loss"] for r in history], marker=".")
axes[0].set(xlabel="Optimizer updates", ylabel="Mean target NLL", title="Training reproduction")
initial_nll = training["initial"]["examples"][0]["nll"]
final_nll = training["final"]["examples"][0]["nll"]
axes[1].plot(initial_nll, label="Before")
axes[1].plot(final_nll, label="After")
axes[1].axvline(SETUP["update_after"] - 0.5, color="black", linestyle="--", label="Image event")
axes[1].set(
    xlabel="Generated target index", ylabel="Token NLL", title="Teacher-forced target logprobs"
)
axes[1].legend()
plt.tight_layout()
fig.savefig(RUN / "training_loss.png", dpi=160)
display_figure(fig)
plt.close(fig)


In [ ]:
checkpoint = RUN / "trained_low_lr/checkpoint"
reload_path = RUN / "checkpoint_reload_low_lr.json"
if not reload_path.exists():
    isolated(
        exp.verify_checkpoint,
        teacher_path,
        checkpoint,
        training_path,
        reload_path,
        log_path=RUN / "checkpoint_reload_low_lr.log",
    )
print(
    "Checkpoint retains training replay:",
    exp.require_replay_options(exp.load(reload_path), SETUP)["passed"],
)
after_path, after = collect_once(checkpoint, "student_after_low_lr")
control_path, control = collect_once(checkpoint, "student_without_update_low_lr", update=False)
for name, run in [("After: image replacement", after), ("After: no image update", control)]:
    display(Markdown(f"### {name}"))
    print(run["assessment"])
    print("Before event:\n", run["before_update_text"])
    print("After event:\n", run["after_update_text"])


In [ ]:
comparison_path = RUN / "comparison_low_lr.json"
if not comparison_path.exists():
    exp.compare(teacher_path, before_path, after_path, control_path, comparison_path)
comparison = exp.require_replay_options(exp.load(comparison_path), SETUP)
columns = [
    "correct_answer",
    "behavior_reproduced",
    "exact_tokens",
    "matching_prefix_tokens",
    "token_sequence_similarity",
    "same_external_events",
]
print("metric".ljust(30), "before".ljust(16), "after".ljust(16), "no update")
for key in columns:
    print(
        key.ljust(30),
        *(
            str(comparison["runs"][name][key]).ljust(16)
            for name in ("before", "after", "without_update")
        ),
    )
print("Free-running behavioral success:", comparison["free_running_success"])
print("Exact full-trajectory reproduction:", comparison["exact_trajectory_reproduction"])
print("Visual counterfactual:", comparison["visual_counterfactual_passed"])


### Same event and notification, unchanged pixels

Replace A with A at the same token barrier and deliver exactly the same correction notice. The expected answer remains +3. This separates the new visual evidence from the mere presence of an update notification.

In [ ]:
pixel_path, pixel_run = collect_once(
    checkpoint, "student_same_image_low_lr", replacement_image="before"
)
pixel_report_path = RUN / "pixel_control_low_lr.json"
if not pixel_report_path.exists():
    exp.compare_pixel_control(teacher_path, after_path, pixel_path, pixel_report_path)
pixel_control = exp.require_replay_options(exp.load(pixel_report_path), SETUP)
print(json.dumps(pixel_control, indent=2))
print(pixel_run["after_update_text"])
assert pixel_control["passed"], "New-pixel dependence has not been established."


The semantic checks are explicit task-specific evidence flags: teacher/student mention the old table value before the event, read the new value afterward, reach the revised answer, and finish. They do not claim to judge arbitrary reasoning equivalence. Compare the phase texts and event records above as well as the token metrics.

The no-update control checks whether the answer follows the image state. The additional A→A control keeps the event and notification identical while withholding the corrected pixels, isolating the visual contribution. Exact token reproduction and the control result are reported independently from behavioral reproduction. A reduced loss alone never marks this experiment successful.

In [ ]:
if not (comparison["free_running_success"] and pixel_control["passed"]):
    display(
        Markdown(
            "**Experiment incomplete:** free-running behavior did not satisfy the explicit trajectory checks. Inspect `comparison.json` and both phase texts; a lower training loss does not establish success."
        )
    )
else:
    display(
        Markdown(
            "**Free-running behavioral reproduction passed** under the fixed async schedule. See the separate exact-token and counterfactual results above."
        )
    )
print("Visual control with identical notification:", pixel_control["passed"])
print("Artifacts:", RUN)
print("Teacher logprob max error:", teacher_parity["parity"]["max_abs_logprob_error"])
print("Student logprob max error:", student_parity["parity"]["max_abs_logprob_error"])
